# 📓 Lab 03 — CNN para FashionMNIST

**Curso:** Deep Learning: de los fundamentos a los Transformers · **Sesión 2 · Laboratorio 2**
**Material del aula:** [Sesión 2 — CNN y visión](../sesiones/02-cnn-vision.md) ·
**Config:** [`configs/cnn.yaml`](../configs/cnn.yaml)

En este laboratorio: convolución en acción, entrenamiento de una CNN completa,
diagnóstico con matriz de confusión y galería de errores, y un experimento
controlado de regularización/optimización.

> ⏱️ En CPU, el entrenamiento completo (12 epochs) tarda ~10–20 min.
> Para iterar rápido, reduce `epochs` o usa un subconjunto.

In [ ]:
from __future__ import annotations

import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

from src.utils import seed_everything, detectar_dispositivo, contar_parametros
from src.data import cargar_fashionmnist
from src.models import FashionCNN
from src.train import entrenar, run_epoch
from src.evaluate import predecir, reporte_completo, errores_alta_confianza

seed_everything(42)
device = detectar_dispositivo()
print('Dispositivo:', device)

## 1. Kernels a mano: la intuición antes de la arquitectura

Antes de que la red APRENDA sus filtros, apliquemos kernels clásicos diseñados
a mano, para ver qué "detecta" una convolución.

In [ ]:
import torch.nn.functional as F
from torchvision import datasets, transforms

# Descarga FashionMNIST (solo la primera vez; queda cacheado en data/)
fashion = datasets.FashionMNIST(root='../data', train=True, download=True,
                                transform=transforms.ToTensor())
imagen, label = fashion[0]          # (1, 28, 28)
print('Clase:', fashion.classes[label])

kernels = {
    'bordes horizontales': torch.tensor([[-1., -1., -1.],
                                         [ 0.,  0.,  0.],
                                         [ 1.,  1.,  1.]]),
    'bordes verticales':   torch.tensor([[-1., 0., 1.],
                                         [-1., 0., 1.],
                                         [-1., 0., 1.]]),
    'sharpen':             torch.tensor([[ 0., -1.,  0.],
                                         [-1.,  5., -1.],
                                         [ 0., -1.,  0.]]),
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(imagen.squeeze(), cmap='gray'); axes[0].set_title('original')
for ax, (nombre, k) in zip(axes[1:], kernels.items()):
    # conv2d espera (batch, canales, H, W) y el kernel (out, in, kH, kW)
    salida = F.conv2d(imagen.unsqueeze(0), k.view(1, 1, 3, 3), padding=1)
    ax.imshow(salida.squeeze(), cmap='gray'); ax.set_title(nombre)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

# Mensaje clave: estos kernels los diseñamos NOSOTROS.
# En la CNN, backpropagation aprenderá los valores óptimos.

## 2. Datos con augmentation (y sin leakage)

`cargar_fashionmnist` ([`src/data.py`](../src/data.py)) aplica augmentation
**solo al train set** y usa un split determinístico 54k/6k.

🧩 *Pregunta:* ¿por qué evaluar con augmentation distorsionaría las métricas?

In [ ]:
train_loader, val_loader, test_loader, class_names = cargar_fashionmnist(
    root='../data', batch_size=128, seed=42
)
print('Clases:', class_names)

imagenes, labels = next(iter(train_loader))
print('batch:', imagenes.shape)     # (128, 1, 28, 28) — NCHW

# Mosaico de ejemplos (con augmentation aplicada)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, img, lab in zip(axes.ravel(), imagenes, labels):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(class_names[lab], fontsize=7)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 3. La arquitectura y su shape tracing

[`src/models.py → FashionCNN`](../src/models.py):
`[Conv→BN→ReLU→Pool] ×2 → Conv→ReLU→AdaptiveAvgPool → Flatten→Dropout→Linear`
(BN = BatchNorm, Sesión 2 §4; AdaptiveAvgPool = pooling que promedia cada feature map a un tamaño fijo, sin importar el tamaño de entrada).

Verifiquemos el contrato de shapes capa por capa — la técnica #1 de debugging.

In [ ]:
model = FashionCNN(dropout=0.25)

# Shape tracing: pasar un batch de prueba y registrar la shape tras cada capa
x = torch.randn(2, 1, 28, 28)
print(f'{"entrada":30s} {tuple(x.shape)}')
for i, capa in enumerate(model.features):
    x = capa(x)
    print(f'{i:2d} {capa.__class__.__name__:27s} {tuple(x.shape)}')
x = model.classifier(x)
print(f'{"clasificador (logits)":30s} {tuple(x.shape)}')

entrenables, totales = contar_parametros(model)
print(f'\nParámetros entrenables: {entrenables:,}')

## 4. Entrenamiento

AdamW + cosine schedule + gradient clipping, 12 epochs. El loop es el mismo
de la Sesión 1 (`src/train.py`) — el loop es el mismo, con dos añadidos:

- **Cosine schedule** (`CosineAnnealingLR`): baja el learning rate siguiendo una curva de coseno (Sesión 2 §5).
- **Gradient clipping** (`grad_clip`): recorta gradientes gigantes para evitar pasos explosivos (lo verás formalmente en la Sesión 3).

In [ ]:
criterion = nn.CrossEntropyLoss()      # recibe LOGITS y labels enteros
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)

model, historia = entrenar(
    model, train_loader, val_loader, criterion, optimizer, device,
    max_epochs=12,
    patience=5,
    scheduler=scheduler,
    grad_clip=5.0,
    verbose_cada=1,
)

In [ ]:
# Curvas de aprendizaje
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(historia.train_loss, label='train')
ax1.plot(historia.val_loss, label='validation')
ax1.set(xlabel='epoch', ylabel='CE loss', title='CNN — pérdida'); ax1.legend()
ax2.plot(historia.train_acc, label='train')
ax2.plot(historia.val_acc, label='validation')
ax2.set(xlabel='epoch', ylabel='accuracy', title='CNN — accuracy'); ax2.legend()
plt.show()

## 5. Evaluación en test + matriz de confusión

¿Qué pares de clases se confunden? (spoiler: `Shirt` es el villano del dataset)

In [ ]:
y_real, y_pred, confianzas = predecir(model, test_loader, device)
reporte_completo(y_real, y_pred, nombres_clases=class_names)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_real, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10), class_names, rotation=45, ha='right')
ax.set_yticks(range(10), class_names)
ax.set_xlabel('predicho'); ax.set_ylabel('real')
ax.set_title('Matriz de confusión normalizada')
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0.01:
            ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center',
                    fontsize=7, color='white' if cm[i, j] > 0.5 else 'black')
fig.colorbar(im); plt.tight_layout(); plt.show()

## 6. Galería de errores de alta confianza

Los errores donde el modelo estaba MÁS seguro son los más informativos:
¿etiqueta dudosa, ambigüedad real o límite del modelo?

In [ ]:
errores = errores_alta_confianza(y_real, y_pred, confianzas, top_k=10)

fig, axes = plt.subplots(2, 5, figsize=(12, 5.5))
for ax, (idx, real, pred, conf) in zip(axes.ravel(), errores):
    imagen, _ = test_loader.dataset[idx]
    ax.imshow(imagen.squeeze(), cmap='gray')
    ax.set_title(f'real: {class_names[real]}\npred: {class_names[pred]}\n'
                 f'conf: {conf:.2f}', fontsize=8)
    ax.axis('off')
plt.suptitle('Top-10 errores por confianza', fontweight='bold')
plt.tight_layout(); plt.show()

# Para cada error pregúntate: ¿el problema es el DATO o el MODELO?

## 7. 🧪 Experimento controlado

Elige UNA hipótesis y cámbiala con **una sola variable**, misma seed y splits:

- Augmentation mejora la generalización.
- BatchNorm permite un learning rate mayor.
- AdamW converge más rápido que SGD+momentum en este presupuesto.
- Dropout alto (0.6) produce underfitting.
- Cosine schedule mejora el mejor checkpoint vs LR constante.

Antes de correr, ejecuta la **prueba de overfit-one-batch** (checklist de
debugging del curso): si el modelo no puede memorizar 32 imágenes, hay un bug.

In [ ]:
# Prueba 'overfit one batch': el detector de bugs más barato que existe
seed_everything(0)
modelo_debug = FashionCNN().to(device)
opt_debug = torch.optim.AdamW(modelo_debug.parameters(), lr=1e-3)
x_small, y_small = next(iter(train_loader))
x_small, y_small = x_small[:32].to(device), y_small[:32].to(device)

modelo_debug.train()
for paso in range(150):
    opt_debug.zero_grad(set_to_none=True)
    loss = criterion(modelo_debug(x_small), y_small)
    loss.backward()
    opt_debug.step()
    if paso % 30 == 0:
        print(f'paso {paso:3d} | loss {loss.item():.4f}')

# Si la loss NO se desploma hacia ~0, revisa datos, loss o arquitectura.

In [ ]:
# ── TU EXPERIMENTO AQUÍ ──────────────────────────────────────────
HIPOTESIS = '...'

# seed_everything(42)
# modelo_b = FashionCNN(dropout=0.6)   # ← ejemplo de variante
# ... reutiliza el flujo de las secciones 4-5 y compara las curvas
#     de ambos runs en la misma gráfica.

## 8. 📝 Entrega

- [ ] Curvas de ambos runs superpuestas
- [ ] Matriz de confusión + classification report del mejor run
- [ ] Galería de 10 errores con comentario por error
- [ ] Conclusión: hipótesis → evidencia → decisión
- [ ] `git commit -m "feat: complete cnn experiment"`

### 🎁 Reto opcional — transfer learning

Fine-tunea `resnet18` (pesos de ImageNet) sobre un subconjunto de CIFAR-10:
congela el backbone, reemplaza `model.fc`, entrena solo la cabeza y compara
contra entrenar desde cero. Guía en la
[Sesión 2 §8](../sesiones/02-cnn-vision.md#8-transfer-learning).